# ГП2 — baseline pose на Waymo (Kaggle)

**Проект:** 2D keypoints пешеходов, driving-домен  
**Автор:** Тургунов Аббос  
**Модель:** YOLOv8n-pose, веса COCO (без fine-tune)  

Это *измерение* baseline, не EDA. Нужны картинки сегмента + `camera_hkp` + `camera_box`.

**Как запустить на Kaggle**
1. New Notebook, GPU (T4).
2. Add data: zip со срезом `training/camera_image`, `camera_box`, `camera_hkp` для одного сегмента (см. `reports/gp2_kaggle.md`).
3. Add-ons → Secrets: имя `WANDB_API_KEY`, значение — ключ с wandb.ai (страница API keys).
4. Runtime → Run All. В конце будет ссылка на run в проекте **pose-av**.

Локально: поправь `WAYMO_ROOT` в следующей ячейке.

In [ ]:
import os, sys, time, json
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
SEGMENT = "10023947602400723454_1120_000_1140_000"
MAX_FRAMES = 40          # None = весь сегмент
IOU_MATCH = 0.3
CONF = 0.35
IMGSZ = 1280             # мелкие пешеходы, 640 мало
MODEL_NAME = "yolov8s-pose.pt"  # n слишком слаб на дальних людях

if IS_KAGGLE:
    # /kaggle/input/<dataset-name>/...
    cands = list(Path("/kaggle/input").glob("**/camera_image"))
    WAYMO_ROOT = cands[0].parents[1] if cands else Path("/kaggle/input")
else:
    WAYMO_ROOT = Path(r"D:/MissingML/pose_estimation/data/waymo_v2")

print("kaggle", IS_KAGGLE, "root", WAYMO_ROOT)


In [ ]:
%pip install -q ultralytics pyarrow opencv-python-headless wandb

## Метрики (Waymo camera-14, как в репозитории)

In [ ]:
import numpy as np
import cv2
import pyarrow.parquet as pq
from collections import defaultdict

CAMERA_ORDER = (1, 5, 13, 6, 14, 7, 15, 8, 16, 9, 17, 10, 18, 19)
CAMERA_SCALES = np.array([0.052, 0.158, 0.158, 0.144, 0.144, 0.124, 0.124,
                          0.214, 0.214, 0.174, 0.174, 0.178, 0.178, 0.158])
COCO_TO_WAYMO = (0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16)

COL_TYPE = "[CameraHumanKeypointsComponent].camera_keypoints[*].type"
COL_X = "[CameraHumanKeypointsComponent].camera_keypoints[*].keypoint_2d.location_px.x"
COL_Y = "[CameraHumanKeypointsComponent].camera_keypoints[*].keypoint_2d.location_px.y"
COL_OCC = "[CameraHumanKeypointsComponent].camera_keypoints[*].keypoint_2d.visibility.is_occluded"
COL_IMG = "[CameraImageComponent].image"
COL_CX = "[CameraBoxComponent].box.center.x"
COL_CY = "[CameraBoxComponent].box.center.y"
COL_SX = "[CameraBoxComponent].box.size.x"
COL_SY = "[CameraBoxComponent].box.size.y"

def coco17_to_waymo14(coco_xy, coco_vis):
    waymo = np.zeros((14, 2), dtype=np.float64)
    vis = np.zeros((14,), dtype=np.float64)
    for dst, src in enumerate(COCO_TO_WAYMO):
        waymo[dst] = coco_xy[src]
        vis[dst] = coco_vis[src]
    if coco_vis[3] > 0 and coco_vis[4] > 0:
        waymo[13] = 0.5 * (coco_xy[3] + coco_xy[4])
        vis[13] = min(coco_vis[3], coco_vis[4])
    else:
        waymo[13] = coco_xy[0]
        vis[13] = coco_vis[0]
    return waymo, vis

def mean_oks(gt_xy, pr_xy, visibility, box_wh):
    s = np.sqrt(max(box_wh[0] * box_wh[1], 1e-12))
    vis = visibility > 0
    if vis.sum() == 0:
        return np.nan
    d2 = np.sum((gt_xy - pr_xy) ** 2, axis=-1)
    denom = 2.0 * (CAMERA_SCALES * s) ** 2
    oks = np.exp(-d2 / np.maximum(denom, 1e-12))
    return float(oks[vis].mean())

def pck(gt_xy, pr_xy, visibility, box_wh, t=0.2):
    s = np.sqrt(max(box_wh[0] * box_wh[1], 1e-12))
    vis = visibility > 0
    if vis.sum() == 0:
        return np.nan
    dist = np.linalg.norm(gt_xy - pr_xy, axis=-1)
    thr = t * CAMERA_SCALES * s
    return float(((dist <= thr) & vis).sum() / vis.sum())

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

## Загрузка среза

In [ ]:
def find_parquet(kind, split="training"):
    hits = list(WAYMO_ROOT.glob(f"**/{kind}/{SEGMENT}.parquet"))
    hits += list(WAYMO_ROOT.glob(f"**/{kind}/**/{SEGMENT}.parquet"))
    if not hits:
        raise FileNotFoundError(kind)
    return hits[0]

img_path = find_parquet("camera_image")
box_path = find_parquet("camera_box")
hkp_path = find_parquet("camera_hkp")
print(img_path)
print(box_path)
print(hkp_path)

img_t = pq.read_table(img_path, columns=[
    "key.frame_timestamp_micros", "key.camera_name", COL_IMG])
box_t = pq.read_table(box_path, columns=[
    "key.frame_timestamp_micros", "key.camera_name", "key.camera_object_id",
    COL_CX, COL_CY, COL_SX, COL_SY])
hkp_t = pq.read_table(hkp_path, columns=[
    "key.frame_timestamp_micros", "key.camera_name", "key.camera_object_id",
    COL_TYPE, COL_X, COL_Y, COL_OCC])
print("images", img_t.num_rows, "boxes", box_t.num_rows, "hkp", hkp_t.num_rows)

In [ ]:
def rows_by_frame(table, extra):
    ts = table.column("key.frame_timestamp_micros").to_pylist()
    cam = table.column("key.camera_name").to_pylist()
    out = defaultdict(list)
    extras = {c: table.column(c).to_pylist() for c in extra}
    n = table.num_rows
    for i in range(n):
        rec = {c: extras[c][i] for c in extra}
        rec["oid"] = None
        out[(ts[i], int(cam[i]))].append(rec)
    return out

imgs = {}
ts_i = img_t.column("key.frame_timestamp_micros").to_pylist()
cam_i = img_t.column("key.camera_name").to_pylist()
blob = img_t.column(COL_IMG).to_pylist()
for i in range(img_t.num_rows):
    imgs[(ts_i[i], int(cam_i[i]))] = blob[i]

boxes = defaultdict(list)
ts_b = box_t.column("key.frame_timestamp_micros").to_pylist()
cam_b = box_t.column("key.camera_name").to_pylist()
oid_b = box_t.column("key.camera_object_id").to_pylist()
cx = box_t.column(COL_CX).to_pylist(); cy = box_t.column(COL_CY).to_pylist()
sx = box_t.column(COL_SX).to_pylist(); sy = box_t.column(COL_SY).to_pylist()
for i in range(box_t.num_rows):
    boxes[(ts_b[i], int(cam_b[i]))].append({
        "oid" : oid_b[i], "cx": cx[i], "cy": cy[i], "w": sx[i], "h": sy[i],
    })

hkps = defaultdict(list)
ts_h = hkp_t.column("key.frame_timestamp_micros").to_pylist()
cam_h = hkp_t.column("key.camera_name").to_pylist()
oid_h = hkp_t.column("key.camera_object_id").to_pylist()
types = hkp_t.column(COL_TYPE).to_pylist()
xs = hkp_t.column(COL_X).to_pylist(); ys = hkp_t.column(COL_Y).to_pylist()
occ = hkp_t.column(COL_OCC).to_pylist()
for i in range(hkp_t.num_rows):
    hkps[(ts_h[i], int(cam_h[i]))].append({
        "oid": oid_h[i], "types": types[i] or [], "x": xs[i] or [],
        "y": ys[i] or [], "occ": occ[i] or [],
    })

keys = [k for k in imgs if k in hkps]
keys.sort()
if MAX_FRAMES is not None:
    keys = keys[:MAX_FRAMES]
print("frames with image+hkp", len(keys))

## Инференс YOLOv8n-pose (COCO pretrained)

In [ ]:
from ultralytics import YOLO
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL_NAME)
print("device", device)

def gt_vec(rec):
    xy = np.zeros((14, 2), dtype=np.float64)
    vis = np.zeros((14,), dtype=np.float64)
    tmap = {int(t): j for j, t in enumerate(rec["types"])}
    for k, t in enumerate(CAMERA_ORDER):
        if t not in tmap:
            continue
        j = tmap[t]
        xy[k] = [rec["x"][j], rec["y"][j]]
        vis[k] = 1
    return xy, vis

def run_full_frame(im):
    t0 = time.perf_counter()
    pred = model.predict(im, verbose=False, device=device, conf=CONF, imgsz=IMGSZ, classes=[0])[0]
    dt = time.perf_counter() - t0
    p_boxes, p_kpts = [], []
    if pred.boxes is not None and pred.keypoints is not None and len(pred.boxes):
        xyxy = pred.boxes.xyxy.cpu().numpy()
        kxy = pred.keypoints.xy.cpu().numpy()
        kconf = pred.keypoints.conf.cpu().numpy() if pred.keypoints.conf is not None else np.ones(kxy.shape[:2])
        for bi in range(len(xyxy)):
            p_boxes.append(xyxy[bi])
            vis = (kconf[bi] > 0.2).astype(np.float64)
            p_kpts.append(coco17_to_waymo14(kxy[bi], vis))
    return pred, p_boxes, p_kpts, dt

def run_on_crop(im, xyxy):
    h, w = im.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in xyxy]
    pad_x, pad_y = int(0.2 * (x2 - x1)), int(0.2 * (y2 - y1))
    x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
    x2, y2 = min(w, x2 + pad_x), min(h, y2 + pad_y)
    if x2 <= x1 + 4 or y2 <= y1 + 4:
        return None
    crop = im[y1:y2, x1:x2]
    pred = model.predict(crop, verbose=False, device=device, conf=0.1, imgsz=256, classes=[0])[0]
    if pred.keypoints is None or len(pred.keypoints) == 0:
        return None
    kxy = pred.keypoints.xy.cpu().numpy()[0].copy()
    kconf = pred.keypoints.conf.cpu().numpy()[0] if pred.keypoints.conf is not None else np.ones(17)
    kxy[:, 0] += x1
    kxy[:, 1] += y1
    vis = (kconf > 0.15).astype(np.float64)
    return coco17_to_waymo14(kxy, vis)

oks_ff, pck_ff, miss_ff, lat = [], [], 0, []
oks_td, pck_td, miss_td = [], [], 0
viz_candidates = []

for fi, key in enumerate(keys):
    jpeg = imgs[key]
    arr = np.frombuffer(jpeg, dtype=np.uint8)
    im = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if im is None:
        continue
    pred, p_boxes, p_kpts, dt = run_full_frame(im)
    lat.append(dt)
    box_by_oid = {b["oid"]: b for b in boxes.get(key, [])}
    frame_gt = []
    max_area = 0.0
    for rec in hkps[key]:
        b = box_by_oid.get(rec["oid"])
        if b is None:
            miss_ff += 1
            miss_td += 1
            continue
        gt_xyxy = [b["cx"]-b["w"]/2, b["cy"]-b["h"]/2, b["cx"]+b["w"]/2, b["cy"]+b["h"]/2]
        gt_xy, gt_vis = gt_vec(rec)
        if gt_vis.sum() == 0:
            continue
        max_area = max(max_area, b["w"] * b["h"])
        frame_gt.append((gt_xyxy, gt_xy, gt_vis, b))
        best_i, best_iou = -1, IOU_MATCH
        for i, pb in enumerate(p_boxes):
            val = iou_xyxy(gt_xyxy, pb)
            if val > best_iou:
                best_iou, best_i = val, i
        if best_i < 0:
            miss_ff += 1
        else:
            pr_xy, _ = p_kpts[best_i]
            wh = np.array([b["w"], b["h"]])
            oks_ff.append(mean_oks(gt_xy, pr_xy, gt_vis, wh))
            pck_ff.append(pck(gt_xy, pr_xy, gt_vis, wh, 0.2))
        crop_pred = run_on_crop(im, gt_xyxy)
        if crop_pred is None:
            miss_td += 1
        else:
            pr_xy, _ = crop_pred
            wh = np.array([b["w"], b["h"]])
            oks_td.append(mean_oks(gt_xy, pr_xy, gt_vis, wh))
            pck_td.append(pck(gt_xy, pr_xy, gt_vis, wh, 0.2))
    viz_candidates.append((max_area, im[:, :, ::-1].copy(), frame_gt, p_boxes, p_kpts))
    if (fi + 1) % 10 == 0:
        print(f"{fi+1}/{len(keys)}  ff_matched={len(oks_ff)} td_matched={len(oks_td)}")

viz_candidates.sort(key=lambda x: -x[0])
viz = viz_candidates[:4]

def pack(name, oks, pck, miss, extra=None):
    d = {
        "protocol": name,
        "matched_instances": len(oks),
        "unmatched_gt": miss,
        "mean_OKS": float(np.nanmean(oks)) if oks else None,
        "PCK@0.2": float(np.nanmean(pck)) if pck else None,
    }
    if extra:
        d.update(extra)
    return d

summary = {
    "model": MODEL_NAME,
    "device": device,
    "imgsz": IMGSZ,
    "conf": CONF,
    "segment": SEGMENT,
    "frames": len(keys),
    "full_frame_bottom_up": pack("full_frame", oks_ff, pck_ff, miss_ff, {
        "latency_ms_mean": float(np.mean(lat)*1000) if lat else None,
        "fps": float(1.0/np.mean(lat)) if lat else None,
    }),
    "gt_crop_top_down": pack("gt_crop", oks_td, pck_td, miss_td),
}
print(json.dumps(summary, indent=2))


In [ ]:
# W&B: те же цифры, что в JSON, уходят на wandb.ai (на Kaggle диск временный)
from kaggle_secrets import UserSecretsClient

def _wandb_key():
    k = os.environ.get("WANDB_API_KEY")
    if k:
        return k
    try:
        return UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        return None

key = _wandb_key()
if not key:
    print("wandb: skip — добавь Secret WANDB_API_KEY (Add-ons → Secrets)")
else:
    os.environ["WANDB_API_KEY"] = key
    import wandb
    run = wandb.init(project="pose-av", name=f"gp2-{MODEL_NAME}-f{len(keys)}", config=summary, finish_previous=True)
    flat = {}
    def walk(d, p=""):
        for k, v in d.items():
            name = f"{p}{k}" if p else k
            if isinstance(v, dict):
                walk(v, name + "/")
            elif isinstance(v, (int, float)) and not isinstance(v, bool):
                flat[name] = float(v)
    walk(summary)
    run.log(flat)
    print("wandb:", run.url)
    run.finish()


In [ ]:
import matplotlib.pyplot as plt

SKELETON = [(0,13),(1,2),(1,3),(3,5),(2,4),(4,6),(1,7),(2,8),(7,8),(7,9),(9,11),(8,10),(10,12)]

def draw_skel(ax, xy, vis, color, lw=1.5):
    for a, b in SKELETON:
        if vis[a] and vis[b]:
            ax.plot([xy[a,0], xy[b,0]], [xy[a,1], xy[b,1]], color=color, lw=lw)
    m = vis > 0
    if m.any():
        ax.scatter(xy[m,0], xy[m,1], s=12, c=color)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, item in zip(axes.ravel(), viz):
    _, rgb, frame_gt, p_boxes, p_kpts = item
    ax.imshow(rgb)
    ax.axis("off")
    for gt_xyxy, gt_xy, gt_vis, b in frame_gt:
        x1, y1, x2, y2 = gt_xyxy
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, ec="red", lw=1.2))
        draw_skel(ax, gt_xy, gt_vis, "red")
    for pb, (pr_xy, pr_vis) in zip(p_boxes, p_kpts):
        ax.add_patch(plt.Rectangle((pb[0], pb[1]), pb[2]-pb[0], pb[3]-pb[1], fill=False, ec="lime", lw=1.0, ls="--"))
        draw_skel(ax, pr_xy, pr_vis, "lime")
plt.suptitle("red = Waymo GT  |  lime = YOLO full-frame  |  кадры с самыми крупными GT-пешеходами")
plt.tight_layout()
plt.show()


## Как читать цифры (ГП2)

- **mean OKS / PCK@0.2** — качество позы на GT-инстансах, которым нашёлся детект IoU≥0.3.
- **unmatched_gt** — GT с keypoints без матча: ошибка детекции, не позы.
- Это COCO-pretrained **без** обучения на Waymo: ожидаем дыру на дальних / спиной / ночью.

### Потолок из литературы (для отчёта)

| Подход | Где меряют | Ориентир |
|---|---|---|
| ViTPose-B/L/H | COCO AP | ~75–79 AP |
| HRNet-W48 | COCO AP | ~75 AP |
| YOLOv8n-pose | COCO pose | сильно ниже больших heatmap, зато real-time |
| Waymo Pose leaderboard | 3D PEM/MPJPE | другой протокол, не сравнивать 1:1 |

Теоретический максимум на *этом* срезе ближе к teacher (ViTPose) после fine-tune, не к COCO AP из бумаги.

### План сетки (ГП4)

1. YOLOv8n vs s vs m, latency vs OKS.
2. Fine-tune на Waymo train hkp.
3. Срезы: camera_id, площадь бокса, число видимых суставов.
4. Teacher ViTPose как потолок и как учитель для distill (ГП5, если успеем).